# 02 · Feature pipeline

Valida y serializa el pipeline compartido de características:

`FeatureUnion` + `CountVectorizer` + `TfidfVectorizer` + `DictVectorizer` + `ToDense` + `Normalize`.

Este notebook parte del cache generado en:

`data/poems_processed.csv`

La regla metodológica es:

- `corpus_role == "reference"`: corpus base usado para ajustar (`fit`) el pipeline.
- `corpus_role == "external"`: corpus externo proyectado con (`transform`) sobre el espacio aprendido.
  
Para una visualización completa, se recomienda usar **nbviewer**:

- [02_feature_pipeline.ipynb (nbviewer)](https://nbviewer.org/github/HubertRonald/VersoVector/blob/main/notebook/02_feature_pipeline.ipynb)

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# Permite importar modules/ y utils/ cuando el notebook se ejecuta desde notebook/
sys.path.insert(0, "..")

%load_ext autoreload
%autoreload 2

# python
import warnings

import numpy as np
import pandas as pd

# sklearn
from sklearn.exceptions import ConvergenceWarning


# modules
from modules.features import build_feature_pipeline, row_l2_norms
from modules.preprocessing import parse_tags
from modules.io import (
    data_path,
    artifact_path,
    load_csv,
    save_csv,
    save_json,
    save_joblib,
)

# typings
from pandas import DataFrame as PandasDF
from typing import Dict, Union, List

# warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore")

# setup
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("max_colwidth", None)

np.set_printoptions(precision=6)

## 1. Cargar corpus procesado

Se carga el archivo generado por `01_cleaning_pipeline.ipynb`.

Este archivo debe contener, como mínimo:

- `title`
- `poet`
- `source`
- `corpus_role`
- `tags`
- `poem_processed`

In [2]:
poems_processed_path: Path = data_path("poems_processed.csv")

poems_df: PandasDF = load_csv(poems_processed_path)

required_columns = {
    "poem_id",
    "title",
    "poet",
    "source",
    "corpus_role",
    "tags",
    "poem_processed",
}

missing_columns = required_columns.difference(poems_df.columns)

if missing_columns:
    raise ValueError(
        f"Faltan columnas requeridas en {poems_processed_path}: {missing_columns}"
    )

poems_df["tags"] = poems_df["tags"].apply(parse_tags)

display(poems_df.head(3))
print(poems_df.shape)

,poem_id,source_row_id,poem_hash,title,title_raw,poet,poet_raw,poem,tags,source,corpus_role,poem_raw,poem_processed
0,poetry_foundation::000000::d1e86366cdbb,0,d1e86366cdbb,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,dog bone stapler cribbage board garlic press because this window is looselacks suction lacks grip bungee cord bootstrap dog leash leather belt because this window had sash cords they frayed they broke feather duster thatch of straw empty bottle of elmers glue because this window is loudits hinges clack open clack shut stuffed bear baby blanket single crib newel because this window is split its dividing in two velvet moss sagebrush willow branch robins wing because this window its paneless its only a frame of air,[],poetry_foundation,reference,"\r\r\nDog bone, stapler,\r\r\ncribbage board, garlic press\r\r\n because this window is loose—lacks\r\r\nsuction, lacks grip.\r\r\nBungee cord, bootstrap,\r\r\ndog leash, leather belt\r\r\n because this window had sash cords.\r\r\nThey frayed. They broke.\r\r\nFeather duster, thatch of straw, empty\r\r\nbottle of Elmer's glue\r\r\n because this window is loud—its hinges clack\r\r\nopen, clack shut.\r\r\nStuffed bear, baby blanket,\r\r\nsingle crib newel\r\r\n because this window is split. It's dividing\r\r\nin two.\r\r\nVelvet moss, sagebrush,\r\r\nwillow branch, robin's wing\r\r\n because this window, it's pane-less. It's only\r\r\na frame of air.\r\r\n",dog bone stapler cribbag board garlic press window looselack suction lack grip bunge cord bootstrap dog leash leather belt window sash cord fray break feather duster thatch straw bottl elmer glue window loudit he clack open clack shut stuf bear babi blanket singl crib newel window split divid velvet moss sagebrush willow branch robin wing window paneless frame air
1,poetry_foundation::000001::0dbfeaec7a94,1,0dbfeaec7a94,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,the old cupola glinted above the clouds shone among fir trees but it took him an hour for the half mile all the way up the hill as he trailed the village passed him by greeted him asked about his health but everybody hurried to catch the mass left him leaning against fences measuring the road with the walking stick he sculpted he yearned for the day when the new church would be builtright across the road now it rises above the moon saints in frescoes meet the eye and only the rain has started to cut through the shingles on the roof of his empty house the apple trees have taken over the sky sequestered the gate sidled over the porch,[],poetry_foundation,reference,"\r\r\nThe old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.\r\r\n",old cupola glint cloud shone fir tree take hour half mile way hill trail villag pass greet ask health everybodi hurri catch mass leave lean fenc measur road walk stick sculpt yearn day new church builtright road rise moon saint fresco meet eye rain start cut shingl roof hous appl tree take sky sequest gate sidl porch
2,poetry_foundation::000002::52eddc66be9e,2,52eddc66be9e,look for me,\r\r\n Look for Me\r\r\n,ted kooser,Ted Kooser,look for me under the hood of that old chevrolet settled in weeds at the end of the pasture im the radiator that spent its years bolted in front of an engine 

(13858, 13)


In [3]:
display(
    pd.DataFrame(
        poems_df[["corpus_role", "source", "poet"]]
        .value_counts()
        .head(20)
    )
)

count
corpus_role source            poet                             
reference   poetry_foundation william shakespeare            85
                              alfred lord tennyson           73
                              william wordsworth             51
                              emily dickinson                51
                              rae armantrout                 49
                              john ashbery                   42
                              yusef komunyakaa               42
                              william butler yeats           41
                              john donne                     38
                              percy bysshe shelley           35
                              algernon charles swinburne     35
                              walt whitman                   35
                              robert browning                35
                              kay ryan                       34
                              edmund spenser                 33
                              samuel menashe                 33
                              william blake                  33
                              john milton                    33
                              henry wadsworth longfellow     32
                              w s di piero                   32

## 2. Separar corpus base y corpus externo

El pipeline se ajusta solamente sobre el corpus de referencia.

Luego, los poemas externos se transforman con el mismo pipeline ya ajustado.

In [4]:
reference_df: PandasDF = (
    poems_df
    .loc[poems_df["corpus_role"].eq("reference")]
    .dropna(subset=["poem_processed"])
    .reset_index(drop=True)
)

external_df: PandasDF = (
    poems_df
    .loc[poems_df["corpus_role"].eq("external")]
    .dropna(subset=["poem_processed"])
    .reset_index(drop=True)
)

if reference_df.empty:
    raise ValueError("No hay poemas con corpus_role == 'reference'.")

print("reference_df:", reference_df.shape)
print("external_df:", external_df.shape)

display(reference_df[["title", "poet", "source", "corpus_role"]].head(3))
display(external_df[["title", "poet", "source", "corpus_role"]].head(3))

reference_df: (13747, 13)
external_df: (4, 13)


,title,poet,source,corpus_role
0,objects used to prop open a window,michelle menting,poetry_foundation,reference
1,the new church,lucia cherciu,poetry_foundation,reference
2,look for me,ted kooser,poetry_foundation,reference


,title,poet,source,corpus_role
0,the black heralds,cesar vallejo,cesar_vallejo,external
1,black stone on top of a white stone,cesar vallejo,cesar_vallejo,external
2,paris october poem,cesar vallejo,cesar_vallejo,external


## 3. Construir y ajustar el feature pipeline

Como `poem_processed` ya fue generado en el notebook 01, se usa:

`build_feature_pipeline(input_is_processed=True)`

Esto evita volver a ejecutar `TokenText()` y, por tanto, evita reprocesar todo con spaCy.

In [5]:
feature_pipeline = build_feature_pipeline(
    input_is_processed=True,
    to_dense=False,
    normalize=True,
)
feature_pipeline

Pipeline(steps=[('Features',
                 FeatureUnion(transformer_list=[('CountVect',
                                                 CountVectorizer(stop_words='english')),
                                                ('Tfid',
                                                 TfidfVectorizer(max_features=5000,
                                                                 norm=None,
                                                                 smooth_idf=False,
                                                                 stop_words='english')),
                                                ('DictVect',
                                                 Pipeline(steps=[('TextToDictTransformer',
                                                                  TextToDictTransformer()),
                                                                 ('DictVectorizer',
                                                                  DictVectorizer())]))])),
                ('Norm', Normalize())])

In [6]:
reference_texts = reference_df["poem_processed"].astype(str).tolist()
external_texts = external_df["poem_processed"].astype(str).tolist()

X_reference = feature_pipeline.fit_transform(reference_texts)

if external_texts:
    X_external = feature_pipeline.transform(external_texts)
else:
    X_external = np.empty((0, X_reference.shape[1]))

print("X_reference shape:", X_reference.shape)
print("X_external shape:", X_external.shape)

X_reference shape: (13747, 223671)
X_external shape: (4, 223671)


## 4. Validaciones rápidas

Como el pipeline termina en `Normalize`, las normas L2 deberían estar cercanas a 1 para filas no vacías.

In [7]:
reference_row_norms = row_l2_norms(X_reference, n_rows=10)
reference_row_norms

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [8]:
if X_external.shape[0] > 0:
    external_row_norms = row_l2_norms(X_external, n_rows=10)
    display(external_row_norms)
else:
    print("No hay corpus externo para validar normas.")

array([1., 1., 1., 1.])

## 5. Metadata de features

Se guarda un resumen del pipeline y las dimensiones generadas.

In [9]:
metadata: Dict[str, Union[str,int,Dict]] = {
    "input_column": "poem_processed",
    "pipeline": (
        "build_feature_pipeline("
        "input_is_processed=True, to_dense=False, normalize=True)"
    ),
    "fit_corpus_role": "reference",
    "transform_corpus_role": "external",
    "n_reference_documents": int(X_reference.shape[0]),
    "n_external_documents": int(X_external.shape[0]),
    "n_features": int(X_reference.shape[1]),
    "reference_sources": reference_df["source"].value_counts().to_dict(),
    "external_sources": external_df["source"].value_counts().to_dict(),
    "to_dense": False,
    "normalize": True,
    "serialized_feature_matrices": False,
    "reason": (
        "Feature matrices can be too large for GitHub/local reload. "
        "They are regenerated from feature_pipeline.joblib."
    ),
}

save_json(
    metadata,
    artifact_path("features", "feature_pipeline_metadata.json"),
)

metadata

{'input_column': 'poem_processed',
 'pipeline': 'build_feature_pipeline(input_is_processed=True, to_dense=False, normalize=True)',
 'fit_corpus_role': 'reference',
 'transform_corpus_role': 'external',
 'n_reference_documents': 13747,
 'n_external_documents': 4,
 'n_features': 223671,
 'reference_sources': {'poetry_foundation': 13747},
 'external_sources': {'cesar_vallejo': 4},
 'to_dense': False,
 'normalize': True,
 'serialized_feature_matrices': False,
 'reason': 'Feature matrices can be too large for GitHub/local reload. They are regenerated from feature_pipeline.joblib.'}

## 6. Guardar artefactos de features

Estos artefactos serán usados por los notebooks posteriores:

- `03_embeddings_supervised.ipynb`
- `04_embeddings_unsupervised.ipynb`
- `05_supervised_unsupervised_integration.ipynb`

In [10]:
save_joblib(
    feature_pipeline,
    artifact_path("features", "feature_pipeline.joblib"),
)

# --------------------------------------------------
# Nota:
# No se guardan X_reference ni X_external como .joblib en el repo porque pueden ser muy pesados.
# Estas matrices se regeneran localmente cuando se ejecutan los notebooks posteriores.
# Para producción futura, se evaluará guardar features reducidas o vecinos precomputados.
"""
save_joblib(
    X_reference,
    artifact_path("features", "reference_features.joblib"),
)

save_joblib(
    X_external,
    artifact_path("features", "external_features.joblib"),
)
"""
# --------------------------------------------------

save_csv(
    reference_df,
    artifact_path("features", "reference_metadata.csv"),
)

save_csv(
    external_df,
    artifact_path("features", "external_metadata.csv"),
)

print("Artefactos de features guardados correctamente.")

Artefactos de features guardados correctamente.


## 7. Validación semántica mínima

Se inspeccionan ejemplos del corpus base y externo para verificar que el split `reference/external` quedó correcto.

In [11]:
column_names: List[str] = [
    "title",
    "poet",
    "source",
    "corpus_role",
    "tags",
]

display(
    reference_df[column_names].head(5)
)

,title,poet,source,corpus_role,tags
0,objects used to prop open a window,michelle menting,poetry_foundation,reference,[]
1,the new church,lucia cherciu,poetry_foundation,reference,[]
2,look for me,ted kooser,poetry_foundation,reference,[]
3,wild life,grace cavalieri,poetry_foundation,reference,[]
4,umbrella,connie wanek,poetry_foundation,reference,[]


In [12]:
display(
    external_df[column_names].head(5)
)

,title,poet,source,corpus_role,tags
0,the black heralds,cesar vallejo,cesar_vallejo,external,[]
1,black stone on top of a white stone,cesar vallejo,cesar_vallejo,external,[]
2,paris october poem,cesar vallejo,cesar_vallejo,external,[]
3,xiii,cesar vallejo,cesar_vallejo,external,[]
